# Notebook 3: 02_data_quality_check
Quality checks and cleaning for simulated data.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sqlite3
from pathlib import Path


In [ ]:
PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / 'data'
RAW_AUDIO = DATA_DIR / 'audio_clip_event.csv'
RAW_DB = DATA_DIR / 'fireworks_monitoring.db'
CLEAN_AUDIO = DATA_DIR / 'audio_clip_event_cleaned.csv'
CLEAN_DB = DATA_DIR / 'fireworks_monitoring_cleaned.db'
TARGET_CONFUSION = {'TP':2418,'FN':376,'FP':534,'TN':5874}

df_raw = pd.read_csv(RAW_AUDIO, parse_dates=['event_time'])
print('Raw shape:', df_raw.shape)
df_raw.head()


In [ ]:
print('Missing values summary:')
print(df_raw.isna().sum().sort_values(ascending=False).head(15))

print()
print('Time range:')
print(df_raw['event_time'].min(), 'to', df_raw['event_time'].max())

print()
print('Scene distribution (raw):')
print(df_raw['scene_type'].value_counts().head(12))

print()
print('Label distribution (raw):')
print(df_raw['binary_label'].value_counts().sort_index())

print()
print('Confusion distribution (raw):')
print(df_raw['error_type'].value_counts())

print()
print('Duplicate clip_id count (raw):', int(df_raw['clip_id'].duplicated().sum()))


In [ ]:
df = df_raw.copy()

scene_map = {
    'residential':'residential_area',
    'residental_area':'residential_area',
    'main_road':'arterial_road',
    'arterialRoad':'arterial_road',
    'construction':'construction_site',
    'site_construction':'construction_site',
    'industrialPark':'industrial_park',
    'industry_park':'industrial_park',
}
df['scene_type'] = df['scene_type'].replace(scene_map)

before_peak = len(df)
df = df[(df['peak_db'] >= 25) & (df['peak_db'] <= 140)].copy()
removed_peak = before_peak - len(df)

mode_by_group = (
    df.dropna(subset=['weather'])
      .groupby(['scene_type','time_bucket'])['weather']
      .agg(lambda x: x.mode().iloc[0])
)

def fill_weather(row):
    if pd.notna(row['weather']):
        return row['weather']
    key = (row['scene_type'], row['time_bucket'])
    if key in mode_by_group.index:
        return mode_by_group[key]
    return 'cloudy'

df['weather'] = df.apply(fill_weather, axis=1)

before_dup = len(df)
df = df.drop_duplicates(subset=['clip_id'], keep='first').copy()
removed_dup = before_dup - len(df)

print('Removed by unreasonable peak:', removed_peak)
print('Removed duplicates:', removed_dup)
print('Cleaned shape:', df.shape)


In [ ]:
print('Missing weather after cleaning:', int(df['weather'].isna().sum()))
print('Duplicate clip_id after cleaning:', int(df['clip_id'].duplicated().sum()))

print()
print('Scene distribution (clean):')
print(df['scene_type'].value_counts())

print()
print('Label distribution (clean):')
print(df['binary_label'].value_counts().sort_index())

cm = df['error_type'].value_counts().to_dict()
print()
print('Confusion distribution (clean):', cm)
aligned = all(cm.get(k,0)==v for k,v in TARGET_CONFUSION.items())
print('Aligned with target confusion:', aligned)


In [ ]:
df.to_csv(CLEAN_AUDIO, index=False)

with sqlite3.connect(RAW_DB) as conn_raw:
    device_info = pd.read_sql_query('SELECT * FROM device_info', conn_raw)
    alert_record = pd.read_sql_query('SELECT * FROM alert_record', conn_raw)
    firework_event_case = pd.read_sql_query('SELECT * FROM firework_event_case', conn_raw)
    threshold_experiment_log = pd.read_sql_query('SELECT * FROM threshold_experiment_log', conn_raw)

with sqlite3.connect(CLEAN_DB) as conn_clean:
    device_info.to_sql('device_info', conn_clean, if_exists='replace', index=False)
    df.to_sql('audio_clip_event', conn_clean, if_exists='replace', index=False)
    alert_record.to_sql('alert_record', conn_clean, if_exists='replace', index=False)
    firework_event_case.to_sql('firework_event_case', conn_clean, if_exists='replace', index=False)
    threshold_experiment_log.to_sql('threshold_experiment_log', conn_clean, if_exists='replace', index=False)

print('Saved cleaned CSV:', CLEAN_AUDIO)
print('Saved cleaned DB:', CLEAN_DB)
